# A Tour of Rust — in a notebook

Runs on the [**evcxr**](https://github.com/evcxr/evcxr) Rust Jupyter kernel
(`rustc 1.97.1`, edition 2024 — same toolchain as this repo's `cargo` targets).

This mirrors the exercises in [`../examples/`](../examples) and the notes in
[`../docs/rust-tutorial.md`](../docs/rust-tutorial.md), but in a form you can
poke at cell by cell.

### How a Rust notebook differs from `examples/NN_name.rs`

| | `examples/*.rs` | this notebook |
|---|---|---|
| entry point | `fn main() { ... }` | no `main` — statements run at top level |
| running | `cargo run --example 01_hello` | Shift+Enter on a cell |
| state | fresh process each run | `let` bindings persist across cells |
| output | only what you `println!` | that, plus the cell's final expression |
| dependencies | `Cargo.toml` | `:dep name = "1.0"` in a cell |

Cells build on each other, so **run them in order** (Kernel → Restart & Run All
after editing something early).

---
## 01 — `println!` and formatting

`println!` is a macro (note the `!`): the format string is parsed at compile
time, so a mismatched placeholder is a compile error, not a runtime surprise.

In [ ]:
println!("Hello, World!");
println!("Hello, {}", "World");           // positional
println!("Hello, {name}", name = "World"); // named
println!("Hello, {0}, {1}, {0}", "World", "Rust"); // indexed, reused

let who = "Rust";
println!("Hello, {who}");                  // captured from scope
println!("{:?}", (1, "two", 3.0));         // Debug formatting
println!("{:>8}|{:<8}|{:^8}|", "right", "left", "mid"); // alignment
println!("{:.3} {:+} {:#x} {:08.2}", 3.14159, 42, 255, 3.5);

#[derive(Debug)]    // This attribute automatically implements the Debug trait for the struct
struct Lang {
    language: String,
    version: String,
}
let rust = Lang {
    language: "rust".to_string(),
    version: "1.80".to_string(),
};
println!("{:?}", rust);
println!("{:#?}", rust);

A notebook bonus: the **last expression** in a cell is displayed, so you can
inspect a value without `println!`. It needs to implement `Debug`.

> **Notebook gotcha:** *"Couldn't automatically determine type of variable `x`.
> Please give it an explicit type."* — evcxr keeps top-level `let` bindings alive
> across cells, and to do that it has to name their type. It can't when the type
> contains a reference, so annotate it (`&'static str`) or use owned data
> (`String`). Bindings inside a `{ ... }` block or a `fn` don't persist, so they
> never hit this.

In [ ]:
// The annotation is for evcxr, not rustc: a persisted binding whose type holds
// a reference needs a nameable lifetime. String literals are already 'static.
let pairs: Vec<(i32, &'static str)> = vec![(1, "one"), (2, "two")];
pairs   // no semicolon -> this is the cell's value, and gets displayed

---
## 02 — Variables: `let`, `mut`, shadowing, `const`

Bindings are **immutable by default**. `mut` opts into mutation; shadowing
re-binds the same name, and can even change the type.

In [ ]:
let x = 5;
println!("x = {x}");

let mut y = 5;
y += 1;
println!("y = {y}");

// Shadowing: a brand-new binding that reuses the name.
let spaces: &'static str = "   ";   // annotated so evcxr can persist it
let spaces = spaces.len();   // &str -> usize, allowed because it's a new binding
println!("spaces = {spaces}");

// `const` must be type-annotated and is inlined at every use site.
const MAX_POINTS: u32 = 100_000;
println!("MAX_POINTS = {MAX_POINTS}");

Scalar types, tuples and arrays. Integers default to `i32`, floats to `f64`.

In [ ]:
let int: i64 = -42;
let float = 2.5_f32;
let boolean = true;
let ch = 'ℝ';                     // char is a Unicode scalar value, 4 bytes

let tup: (i32, f64, char) = (500, 6.4, 'z');
let (a, b, c) = tup;              // destructuring
println!("{int} {float} {boolean} {ch} | {a} {b} {c} | tup.0 = {}", tup.0);

let arr = [1, 2, 3, 4, 5];        // [i32; 5] — fixed length, on the stack
let zeros = [0u8; 4];             // [0, 0, 0, 0]
println!("{arr:?} {zeros:?} len={} first={}", arr.len(), arr[0]);

---
## 03 — Ownership and borrowing

The concept that most differs from other languages, which is why it comes
before functions in this repo's ordering. Three rules:

1. Every value has exactly one **owner**.
2. When the owner goes out of scope, the value is dropped.
3. There can be *either* one `&mut` reference *or* any number of `&` references
   to a value at a time — never both.

In [ ]:
// Heap data MOVES on assignment — s1 is no longer usable afterwards.
let s1 = String::from("hello");
let s2 = s1;
println!("s2 = {s2}");
// println!("{s1}");   // <- uncomment: error[E0382]: borrow of moved value `s1`

// `.clone()` when you genuinely want a second copy of the heap data.
let s3 = String::from("hello");
let s4 = s3.clone();
println!("s3 = {s3}, s4 = {s4}");

// Stack-only types are `Copy` — no move, both stay valid.
let n1 = 5;
let n2 = n1;
println!("n1 = {n1}, n2 = {n2}");

In [ ]:
// Borrowing: pass a reference and the caller keeps ownership.
fn calculate_length(s: &String) -> usize {
    s.len()
}

let s = String::from("hello");
println!("length of '{s}' is {}", calculate_length(&s));  // s still owned here

// &mut lets the callee modify the original.
fn append_world(s: &mut String) {
    s.push_str(", world");
}

let mut greeting = String::from("hello");
append_world(&mut greeting);
println!("{greeting}");

In [ ]:
// String slices: a borrowed view (&str) into part of a String — no copying.
let sentence = String::from("hello wonderful world");

// The slices borrow `sentence`, so they live in a block: a borrow's type has no
// lifetime evcxr can name, and only top-level bindings persist across cells.
{
    let first: &str = &sentence[0..5];
    let last = &sentence[16..];
    println!("{first:?} .. {last:?}");
}

// Which is why &str is the idiomatic parameter type: it accepts both
// a &String (via deref coercion) and a string literal.
fn shout(s: &str) -> String {
    s.to_uppercase()
}
println!("{} / {}", shout(&sentence), shout("literal"));

---
## 04 — Functions: expressions vs. statements

Rust is expression-oriented. A block's **final expression without a semicolon**
is its value — that's how functions return without the `return` keyword.

In [ ]:
fn add(a: i32, b: i32) -> i32 {
    a + b        // no semicolon => this is the return value
}

fn add_explicit(a: i32, b: i32) -> i32 {
    return a + b;    // works, but reserved by convention for early returns
}

// A block is itself an expression.
let y = {
    let x = 3;
    x + 1        // block evaluates to 4
};

println!("{} {} {y}", add(2, 3), add_explicit(2, 3));

In [ ]:
// Unit type: no `-> T` means the function returns `()`.
fn log(msg: &str) {
    println!("[log] {msg}");
}
log("functions can return nothing");

// Multiple values come back as a tuple.
fn min_max(xs: &[i32]) -> (i32, i32) {
    let mut lo = xs[0];
    let mut hi = xs[0];
    for &x in xs {
        if x < lo { lo = x; }
        if x > hi { hi = x; }
    }
    (lo, hi)
}
let (lo, hi) = min_max(&[3, 9, -1, 7]);
println!("lo = {lo}, hi = {hi}");

---
## 05 — Control flow

`if`, `match` and `loop` are all **expressions**, so they can produce values.

In [ ]:
let number = 6;

// if/else if/else — the condition must be a bool, no truthiness.
if number % 4 == 0 {
    println!("divisible by 4");
} else if number % 3 == 0 {
    println!("divisible by 3");
} else {
    println!("not divisible by 4 or 3");
}

// if as an expression (both arms must have the same type).
let label = if number % 2 == 0 { "even" } else { "odd" };
println!("{number} is {label}");

In [ ]:
// `loop` + `break value` — the only loop that can produce a value.
let mut counter = 0;
let result = loop {
    counter += 1;
    if counter == 10 {
        break counter * 2;
    }
};
println!("loop result = {result}");

// while
let mut n = 3;
while n != 0 {
    print!("{n}... ");
    n -= 1;
}
println!("liftoff!");

// for over a collection, and over a range
for element in [10, 20, 30] {
    print!("{element} ");
}
println!();
for i in (1..4).rev() {
    print!("{i} ");
}
println!();

In [ ]:
// match is exhaustive — the compiler rejects a missing case.
fn describe(n: i32) -> &'static str {
    match n {
        0 => "zero",
        1..=9 => "single digit",
        n if n < 0 => "negative",
        _ => "big",
    }
}
for n in [-5, 0, 7, 42] {
    println!("{n:>3} -> {}", describe(n));
}

---
## 06 — Collections: `Vec`, `String`, `HashMap`

Growable, heap-allocated, and owned by the binding that holds them.

In [ ]:
// Vec<T> — built here with new() + push() to show the mutating API.
let mut v: Vec<i32> = Vec::new();
v.push(1);
v.push(2);
v.push(3);
println!("{v:?}  len={}  sum={}", v.len(), v.iter().sum::<i32>());

// Indexing panics out of bounds; .get() returns Option<&T> instead.
println!("v[0] = {}", v[0]);
println!("v.get(99) = {:?}", v.get(99));

// The vec![] macro is the everyday shorthand.
let v2 = vec![1, 2, 3];
println!("{v2:?}");

In [ ]:
// String is a growable, UTF-8 encoded, owned string.
let mut s = String::from("Hello");
s.push_str(", world");
s.push('!');
println!("{s}  ({} bytes, {} chars)", s.len(), s.chars().count());

// Concatenation: format! borrows everything and is usually clearest.
let hello = String::from("Hello");
let name = String::from("Rust");
println!("{}", format!("{hello}, {name}!"));

// Indexing by integer is NOT allowed (UTF-8 is variable width) — iterate.
for (i, c) in "héllo".char_indices() {
    print!("({i},{c}) ");
}
println!();

In [ ]:
use std::collections::HashMap;

let mut scores: HashMap<String, i32> = HashMap::new();
scores.insert(String::from("Blue"), 10);
scores.insert(String::from("Yellow"), 50);

// get() -> Option<&V>
println!("Blue = {:?}", scores.get("Blue"));

// entry().or_insert() — insert only if absent, returns &mut V
*scores.entry(String::from("Blue")).or_insert(0) += 5;
scores.entry(String::from("Red")).or_insert(1);

// Iteration order is UNSPECIFIED — sort if you need determinism.
// `items` holds references into `scores`, so it stays inside a block.
{
    let mut items: Vec<_> = scores.iter().collect();
    items.sort();
    for (team, score) in items {
        println!("{team}: {score}");
    }
}

---
## 07 — Structs, `impl`, methods

`impl` blocks hold both **methods** (take `self`) and **associated functions**
(don't — `String::from` is one).

In [ ]:
#[derive(Debug, Clone, PartialEq)]
struct Rectangle {
    width: u32,
    height: u32,
}

impl Rectangle {
    // Associated function — called as Rectangle::square(3)
    fn square(size: u32) -> Self {
        Self { width: size, height: size }
    }

    // Method — &self borrows, so the caller keeps the struct
    fn area(&self) -> u32 {
        self.width * self.height
    }

    fn can_hold(&self, other: &Rectangle) -> bool {
        self.width > other.width && self.height > other.height
    }

    // &mut self to mutate in place
    fn scale(&mut self, factor: u32) {
        self.width *= factor;
        self.height *= factor;
    }
}

let mut rect = Rectangle { width: 30, height: 50 };
let sq = Rectangle::square(3);

println!("{rect:?}");
println!("area = {}", rect.area());
println!("can_hold(square) = {}", rect.can_hold(&sq));

rect.scale(2);
println!("scaled = {rect:?}");
println!("{:#?}", sq);   // {:#?} is pretty-printed Debug

In [ ]:
// Tuple structs and unit structs.
struct Point(i32, i32);
struct Marker;

let p = Point(3, 4);
println!("({}, {})", p.0, p.1);

// Struct update syntax: take the rest of the fields from another value.
let base = Rectangle { width: 1, height: 1 };
let taller = Rectangle { height: 10, ..base.clone() };
println!("{taller:?}");
let _ = Marker;

---
## 08 — Enums, `Option`, pattern matching

Rust enums are **sum types**: each variant can carry its own data. `Option` and
`Result` are just enums from the standard library — which is why they come
before traits in this repo's ordering.

In [ ]:
#[derive(Debug)]
enum Message {
    Quit,                       // no data
    Move { x: i32, y: i32 },    // named fields, like a struct
    Write(String),              // a single value
    ChangeColor(u8, u8, u8),    // a tuple
}

fn handle(msg: &Message) -> String {
    match msg {
        Message::Quit => "quit".to_string(),
        Message::Move { x, y } => format!("move to ({x}, {y})"),
        Message::Write(text) => format!("write {text:?}"),
        Message::ChangeColor(r, g, b) => format!("color #{r:02x}{g:02x}{b:02x}"),
    }
}

for msg in [
    Message::Quit,
    Message::Move { x: 3, y: 7 },
    Message::Write(String::from("hi")),
    Message::ChangeColor(255, 128, 0),
] {
    println!("{:<40} -> {}", format!("{msg:?}"), handle(&msg));
}

In [ ]:
// Option<T> replaces null: Some(T) or None, and the compiler makes you handle both.
let some_number: Option<i32> = Some(5);
let no_number: Option<i32> = None;

fn plus_one(x: Option<i32>) -> Option<i32> {
    match x {
        Some(i) => Some(i + 1),
        None => None,
    }
}
println!("{:?} {:?}", plus_one(some_number), plus_one(no_number));

// if let — a match with only one interesting arm
if let Some(n) = some_number {
    println!("got {n}");
}

// let else — bind or bail out
fn double_or_zero(x: Option<i32>) -> i32 {
    let Some(n) = x else { return 0 };
    n * 2
}
println!("{} {}", double_or_zero(Some(21)), double_or_zero(None));

// Common combinators beat hand-written matches
println!("{} {:?}", some_number.unwrap_or(0), some_number.map(|n| n * 10));

---
## 09 — Traits and generics

A trait is a set of methods a type promises to provide (an interface).
Generics are monomorphized at compile time, so there's no dispatch cost.

In [ ]:
use std::fmt::Display;

trait Summary {
    fn summarize_author(&self) -> String;

    // Default method — implementors may override it.
    fn summarize(&self) -> String {
        format!("(Read more from {}...)", self.summarize_author())
    }
}

struct Tweet { username: String, content: String }
struct Article { headline: String, author: String }

impl Summary for Tweet {
    fn summarize_author(&self) -> String { format!("@{}", self.username) }
    fn summarize(&self) -> String { format!("{}: {}", self.summarize_author(), self.content) }
}

impl Summary for Article {
    fn summarize_author(&self) -> String { self.author.clone() }
    // uses the default summarize()
}

let tweet = Tweet { username: "rustlang".into(), content: "1.97 is out".into() };
let article = Article { headline: "Ownership".into(), author: "Steve".into() };
println!("{}", tweet.summarize());
println!("{} — {}", article.headline, article.summarize());

In [ ]:
// Generic function with a trait bound: T must be comparable.
fn largest<T: PartialOrd + Copy>(list: &[T]) -> T {
    let mut largest = list[0];
    for &item in list {
        if item > largest {
            largest = item;
        }
    }
    largest
}
println!("{} {} {}", largest(&[34, 50, 25]), largest(&[1.5, 0.2]), largest(&['y', 'm']));

// `impl Trait` in argument position is sugar for a bound.
fn notify(item: &impl Summary) {
    println!("Breaking! {}", item.summarize());
}
notify(&tweet);

// where-clauses keep long signatures readable.
fn describe_pair<T, U>(a: T, b: U) -> String
where
    T: Display,
    U: Display,
{
    format!("{a} and {b}")
}
println!("{}", describe_pair(1, "two"));

In [ ]:
// Generic struct + an impl block restricted to one concrete type.
#[derive(Debug)]
struct Wrapper<T> { value: T }

impl<T: Display> Wrapper<T> {
    fn show(&self) -> String { format!("[{}]", self.value) }
}

impl Wrapper<f64> {
    fn rounded(&self) -> i64 { self.value.round() as i64 }
}

println!("{} {}", Wrapper { value: "hi" }.show(), Wrapper { value: 2.7 }.rounded());

---
## 10 — Error handling: `Result` and `?`

`panic!` is for unrecoverable bugs; `Result<T, E>` is for expected failure.

> **Notebook gotcha:** `?` only works inside a function that returns
> `Result`/`Option`, and notebook cells are *not* such a function — so wrap
> `?`-using code in a small `fn` and call it, as below.

In [ ]:
#[derive(Debug)]
enum MathError {
    DivideByZero,
    NegativeSqrt(f64),
}

fn divide(a: f64, b: f64) -> Result<f64, MathError> {
    if b == 0.0 {
        Err(MathError::DivideByZero)
    } else {
        Ok(a / b)
    }
}

println!("{:?}", divide(10.0, 2.0));
println!("{:?}", divide(10.0, 0.0));

// Matching on a Result
match divide(1.0, 0.0) {
    Ok(v) => println!("ok: {v}"),
    Err(e) => println!("failed: {e:?}"),
}

In [ ]:
fn sqrt(x: f64) -> Result<f64, MathError> {
    if x < 0.0 { Err(MathError::NegativeSqrt(x)) } else { Ok(x.sqrt()) }
}

// `?` unwraps Ok, or returns early with the Err — the whole point of Result ergonomics.
fn sqrt_of_quotient(a: f64, b: f64) -> Result<f64, MathError> {
    let q = divide(a, b)?;
    let r = sqrt(q)?;
    Ok(r)
}

println!("{:?}", sqrt_of_quotient(8.0, 2.0));
println!("{:?}", sqrt_of_quotient(8.0, 0.0));
println!("{:?}", sqrt_of_quotient(-8.0, 2.0));

In [ ]:
// Boxing errors so different error types can flow through one function.
fn parse_and_double(s: &str) -> Result<i32, Box<dyn std::error::Error>> {
    let n: i32 = s.trim().parse()?;   // ParseIntError converts into Box<dyn Error>
    Ok(n * 2)
}

println!("{:?}", parse_and_double(" 21 "));
match parse_and_double("abc") {
    Ok(v) => println!("ok {v}"),
    Err(e) => println!("err: {e}"),
}

// unwrap / expect panic on Err — fine in a notebook or a test, avoid in libraries.
println!("{}", "7".parse::<i32>().expect("should be a number"));

---
## 11 — Closures and iterators

Closures capture their environment; iterators are **lazy** and do nothing until
consumed by something like `collect`, `sum` or `for`.

In [ ]:
// Closures infer their types; |x| x + 1 is the whole syntax.
let add_one = |x: i32| x + 1;
println!("{}", add_one(41));

// Capturing by reference...
let factor = 3;
let scale = |x: i32| x * factor;
println!("{}", scale(5));

// ...by mutable reference...
let mut count = 0;
let mut bump = || { count += 1; };
bump();
bump();
println!("count = {count}");

// ...or by move (takes ownership — needed for threads / returning closures).
let owned = String::from("captured");
let show = move || println!("{owned}");
show();

// Returning a closure
fn adder(n: i32) -> impl Fn(i32) -> i32 {
    move |x| x + n
}
println!("{}", adder(10)(5));

In [ ]:
let numbers = vec![1, 2, 3, 4, 5, 6, 7, 8, 9, 10];

let doubled: Vec<i32> = numbers.iter().map(|n| n * 2).collect();
// .iter() yields &i32, so filter's closure sees &&i32 — hence the *n.
// .copied() turns the &i32 back into i32; collecting Vec<&i32> would work in a
// program, but a persisted notebook binding can't hold borrows.
let evens: Vec<i32> = numbers.iter().filter(|n| *n % 2 == 0).copied().collect();
let sum: i32 = numbers.iter().sum();
// fold is the general reduce; .product() is the shortcut for this specific case.
let product: i64 = numbers.iter().fold(1i64, |acc, &n| acc * n as i64);

println!("doubled  = {doubled:?}");
println!("evens    = {evens:?}");
println!("sum      = {sum}, product = {product}");

// Chaining, with index and early termination
let report: Vec<String> = numbers
    .iter()
    .enumerate()
    .filter(|(_, n)| *n % 3 == 0)
    .map(|(i, n)| format!("#{i}={n}"))
    .collect();
println!("{}", report.join(", "));

println!("any >9?  {}", numbers.iter().any(|n| *n > 9));
println!("all >0?  {}", numbers.iter().all(|n| *n > 0));
println!("first>4: {:?}", numbers.iter().find(|n| **n > 4));
println!("take 3:  {:?}", numbers.iter().take(3).collect::<Vec<_>>());
println!("zip:     {:?}", numbers.iter().zip("abc".chars()).collect::<Vec<_>>());

---
## Notebook-only extras

Things you can do here that a `cargo run --example` can't.

### Pulling in a crate with `:dep`

`:dep` compiles the crate into the kernel's session — the first one in a fresh
kernel takes a while, then it's cached. Nothing is written to this repo's
`Cargo.toml`.

```rust
:dep rand = "0.8"
use rand::Rng;
let n: u8 = rand::thread_rng().gen_range(1..=6);
println!("rolled {n}");
```

(Left as a Markdown cell so *Run All* stays fast and offline. Copy it into a
code cell when you want it.)

### Kernel commands

| command | what it does |
|---|---|
| `:help` | list all commands |
| `:vars` | show the variables currently defined in the session |
| `:dep name = "1.0"` | add a crate to the session |
| `:opt 2` | turn on optimisation (default is `0`, fast to compile) |
| `:timing` | report how long each cell takes |
| `:clear` | drop all defined variables |
| `:last_error_json` | the last error as JSON |

In [ ]:
:vars

---
## Where to go next

- The same material as runnable programs: [`../examples/`](../examples) —
  `cargo run --example 08_enums_pattern_matching`
- Repo-specific gotchas: [`../docs/language-notes.md`](../docs/language-notes.md)
- Cargo commands: [`../CARGO_CHEATSHEET.md`](../CARGO_CHEATSHEET.md)
- [The Rust Book](https://doc.rust-lang.org/book/) and
  [Rust by Example](https://doc.rust-lang.org/rust-by-example/)